# 12. Тестирование Span-based RuBERT NER

Ноутбук читает threshold, выбранный на validation, один раз оценивает test и сохраняет метрики, predictions и фактически использованную decoding-конфигурацию.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/span_ner_corrected_v1.yaml'
OUTPUT_DIR = PROJECT_DIR / 'results/span_ner_corrected_v1/seed_42'
CHECKPOINT = OUTPUT_DIR / 'checkpoints/best'
CALIBRATION = OUTPUT_DIR / 'threshold_calibration.json'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'

for required_path in (EXPERIMENT_CONFIG, CHECKPOINT, CALIBRATION, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(
            f'Не найден {required_path}. Выполните ноутбуки 10 и 11 по порядку.'
        )

BEST_THRESHOLD = float(json.loads(CALIBRATION.read_text(encoding='utf-8'))['best_threshold'])
print(f'Зафиксированный validation threshold: {BEST_THRESHOLD:.2f}')

bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
from rurebus_ie.training import test_span_ner_experiment

result = test_span_ner_experiment(
    EXPERIMENT_CONFIG,
    project_root=PROJECT_DIR,
    checkpoint_dir=CHECKPOINT,
    confidence_threshold_override=BEST_THRESHOLD,
)
print(f'Test strict micro-F1: {result.metrics.micro_f1:.4f}')
print(f'Test strict macro-F1: {result.metrics.macro_f1:.4f}')
print(f'Precision: {result.metrics.precision:.4f}')
print(f'Recall: {result.metrics.recall:.4f}')

In [ ]:
import pandas as pd

per_class = pd.DataFrame(result.metrics.per_class).T.sort_values('f1')
display(per_class)
per_class['f1'].plot.bar(ylim=(0, 1), grid=True, title='Span NER: test F1 по типам сущностей');
print('Использованный decoding config:', OUTPUT_DIR / 'test_decoding_config.json')